# 🌌 The Masterclass: UMAP (Uniform Manifold Approximation & Projection)

Welcome to the absolute bleeding edge of Bioinformatics.

If you read modern papers on **Single-Cell RNA Sequencing (scRNA-seq)** or **Flow Cytometry**, you will never see PCA. You will occasionally see t-SNE. Almost entirely, you will see **UMAP**.

### Why does UMAP dominate modern biology?
1. **PCA** is linear. It crushes data by drawing straight lines (Eigenvectors). If your biological data is twisted like a DNA helix, straight lines fail completely.
2. **t-SNE** is non-linear and creates gorgeous plots, but it destroys *global distance*. If a T-Cell cluster is plotted next to a B-Cell cluster in t-SNE, it doesn't actually mean they are genetically related. It also takes extremely long to process millions of cells.
3. **UMAP** preserves BOTH local cell types AND global biological lineage! If two clusters are near each other in UMAP, they evolved from the same stem cell tree. Plus, UMAP is blisteringly fast due to advanced topological math.

In [ ]:
# UMAP is so advanced that it is NOT built into standard scikit-learn!
# ⚠️ IF THIS CELL FAILS: Open your terminal and run: `pip install umap-learn`

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import umap.umap_ as umap_library
    print("UMAP Library loaded successfully! Prepare for topology.")
except ImportError:
    print("\n❌ CRITICAL: UMAP is not installed! \nOpen your terminal and type: pip install umap-learn")
    
from sklearn.datasets import load_digits
sns.set_theme(style="darkgrid")

---
## 🔬 Part 1: The High-Dimensional Problem
To truly test UMAP, 3D data isn't enough. We need massive dimensions.

We will use the **Digits dataset** (64 Dimensions). Imagine this as 64 different gene expressions measured across thousands of distinct cells. Our goal is to crush these 64 genes down into exactly 2 dimensions so we can visualize the "Cell Types" on my monitor.

In [ ]:
# Load 64-Dimensional Data
data_bunch = load_digits()
X_high_dim = data_bunch.data
labels = data_bunch.target # True biological classifications (0 through 9)

print(f"Raw Data Shape: {X_high_dim.shape}")
print("That means 1,797 patients, each with 64 distinct numerical features.")

---
## 🧮 Part 2: The Two Master Hyperparameters of UMAP

Unlike PCA where you just run `.fit()`, UMAP builds a physical "web" connecting data points using **Fuzzy Simplicial Sets** (heavy algebraic topology). You must dictate exactly how the web is woven using two parameters:

### 1. `n_neighbors` (The Web Span)
* **Low (e.g., 5):** The AI only connects cells to their absolute closest neighbors. It focuses heavily on extreme local structure. Your graph will look like hundreds of tiny fragmented islands.
* **High (e.g., 100):** The AI connects cells across vast distances. It considers the entire dataset at once. You get massive generalized blobs.
* **Golden Standard:** 15. This expertly balances the big picture with the small details.

### 2. `min_dist` (The Clumpiness)
Once UMAP figures out the 2D layout, how tightly should it pack the cells together visually?
* **Low (e.g., 0.0):** Allowed to physically stack cells directly on top of each other. Produces incredibly tight structural clusters.
* **High (e.g., 0.8):** Cells forcefully push each other away. Produces loose, spread-out clouds.
* **Golden Standard for Bioinformatics:** 0.1 (We want to see distinct sharp clusters!).

In [ ]:
# Instantiate the UMAP Mathematical Engine
umap_engine = umap_library.UMAP(
    n_neighbors=15,    # Balance between local and global grouping
    min_dist=0.1,      # Pack them tightly together
    n_components=2,    # Crush 64 dimensions into 2!
    metric='euclidean',# How we calculate distance
    random_state=42    # Reproducibility
)

# Engage the algorithm
print("Engaging UMAP Topology mapping...")
X_umap_2D = umap_engine.fit_transform(X_high_dim)
print(f"Success! Crushed from {X_high_dim.shape[1]}D down to {X_umap_2D.shape[1]}D.")

---
## 🖼️ Part 3: Visualizing the Topology
Let's see if UMAP autonomously found the 10 real classes hidden inside the 64-dimensional space without us giving it the labels during training.

In [ ]:
plt.figure(figsize=(10, 8))

# Plot the UMAP coordinates
scatter = sns.scatterplot(
    x=X_umap_2D[:, 0], 
    y=X_umap_2D[:, 1], 
    hue=labels, 
    palette="Spectral", 
    alpha=0.8, 
    s=30, 
    edgecolor='none'
)

plt.title("UMAP Projection of 64-Dimensional Space", fontsize=16, fontweight='bold')
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")

# Clean up the legend
plt.legend(title="True Class", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


### 🎉 Conclusion: Why UMAP won.

Look at the graph above. All 10 clusters are vividly distinct.

Unlike PCA (which would look like a giant blurred ball in the center), UMAP violently separated the "Species". But unlike t-SNE, the distances *between* the clusters actually matter mathematically. The clusters closest to each other represent elements that share fundamental similarities in high-dimensional space.

If you ever sequence 100,000 cells to find a novel biomarker, this is exactly the algorithm you will run on Day 1.